In [1]:
import os
import snowflake.connector
from pathlib import Path
import re
from dotenv import load_dotenv

In [7]:

PLACEHOLDER = re.compile(r"\{\{\s*([A-Z_][A-Z0-9_]*)\s*\}\}")
IDENTIFICADOR_SEGURO = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*$")

def renderizar_sql(texto: str) -> str:
    missing = set()

    def substituir(m):
        name = m.group(1)
        valor = os.environ.get(name)
        if valor is None:
            missing.add(name)
            return m.group(0)
        if not IDENTIFICADOR_SEGURO.match(valor):
            raise ValueError(f"Valor inválido para {name}: {valor!r}")
        return valor

    result = PLACEHOLDER.sub(substituir, texto)
    if missing:
        raise KeyError(f"Variáveis de ambiente não definidas: {sorted(missing)}")
    return result

sql = renderizar_sql(Path("00_infrastructure_commands.sql").read_text(encoding="utf-8"))

load_dotenv()  # Carrega variáveis de ambiente do arquivo .env

with snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA']
) as conn:
    cursors = conn.execute_string(sql, remove_comments=True)
    for cur in cursors:
        print(cur.query.strip()[:80], "->", cur.fetchall())

CREATE WAREHOUSE IF NOT EXISTS AVIATION_WEATHER_WAREHOUSE WAREHOUSE_SIZE = XSMAL -> [('AVIATION_WEATHER_WAREHOUSE already exists, statement succeeded.',)]
CREATE DATABASE IF NOT EXISTS AVIATION_WEATHER_PROJECT; -> [('AVIATION_WEATHER_PROJECT already exists, statement succeeded.',)]
CREATE SCHEMA IF NOT EXISTS AVIATION_WEATHER_PROJECT.RAW_DATA; -> [('RAW_DATA already exists, statement succeeded.',)]
CREATE STAGE IF NOT EXISTS AVIATION_WEATHER_PROJECT.RAW_DATA.WEATHER_DATA 
	URL  -> [('WEATHER_DATA already exists, statement succeeded.',)]
CREATE STAGE IF NOT EXISTS AVIATION_WEATHER_PROJECT.RAW_DATA.BTS_FLIGHTS
DIRECTO -> [('BTS_FLIGHTS already exists, statement succeeded.',)]
GRANT USAGE ON INTEGRATION git_api_integration TO ROLE ACCOUNTADMIN; -> [('Statement executed successfully.',)]
GRANT USAGE ON SECRET git_secret TO ROLE ACCOUNTADMIN; -> [('Statement executed successfully.',)]
CREATE OR REPLACE SECRET git_secret
TYPE = password
USERNAME = 'victorlos'
PASSW -> [('Secret GIT_SECRET su